# COMP4153 Natural Language Processing
# Lab 4 — Sequence Models for Text Classification: RNN, LSTM and GRU

**Version:** Student  
**Duration:** approximately 2 hours  
**Environment:** Google Colab or local Jupyter Notebook  
**Mode:** self-guided — read the explanations and run the cells in order.

This lab applies **sequence models to binary sentiment classification** using exactly the same two datasets and overall activity structure as Lab 3.

- **Part A — Guided IMDB example:** build and train a recurrent classifier, then inspect its learning curves, performance and mistakes.
- **Part B — Twitter practice:** reuse the **same** model and helper functions on shorter tweets, then complete four focused tasks.


```text
text → tokens → training vocabulary → padded IDs + true lengths
→ word embeddings → packed RNN / LSTM / GRU → final hidden state
→ dropout → 2 sentiment logits → cross-entropy
                             ↓
                   backpropagation through time (BPTT)
```

## Learning objectives

By the end of the lab, you should be able to:

- explain why a recurrent model can use **word order** whereas mean pooling cannot;
- distinguish a basic RNN, LSTM and GRU at the level of their hidden-state interfaces;
- convert reviews and tweets to token IDs, right padding and true sequence lengths;
- describe why `pack_padded_sequence()` prevents updates on padding tokens;
- trace tensor shapes through embeddings, recurrent states and classification logits;
- use **one** configurable network and one training function for all three cell types;
- identify the forward pass, cross-entropy loss, BPTT, gradient clipping and optimiser step;
- finish all configured epochs and then restore the lowest-development-loss completed checkpoint;
- interpret learning curves, test accuracy, macro F1, confusion matrices and classification mistakes.

## Suggested timing

| Part | Approx. time |
|---|---:|
| Setup | 5 min |
| Part A — Guided IMDB sequence classifier | 70 min |
| Part B — Guided Twitter preparation and four tasks | 45 min |

# 0. Setup

The lab uses the **same local CSV files** as Lab 3:

- `imdb_reviews_10000_fixed_split.csv` — 10,000 balanced IMDB reviews, with **8,000 train / 1,000 development / 1,000 test** rows;
- `twitter_sentiment_fixed_split.csv` — Twitter sentiment data, with its **supplied train / development / test** splits. The provided `clean_text` column will be used for modelling; `text` is retained for error inspection.

The IMDB subset originates from the [IMDB Movie Review Dataset](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews).

**Do not recreate, resample or reshuffle the supplied split labels.** Each vocabulary is built using *training text only*.

This notebook runs locally on CPU or on a CUDA GPU. It requires **PyTorch, NumPy, pandas, Matplotlib, NLTK and scikit-learn**, as in Lab 3.

## 0.1 Import the libraries

- `numpy` / `pandas` — arrays, tables and experiment histories;
- `TreebankWordTokenizer` / `TweetTokenizer` — IMDB/Twitter tokenisation;
- `torch` — embeddings, recurrent layers and training;
- `pack_padded_sequence` — ignore trailing `<PAD>` positions;
- `scikit-learn` / Matplotlib — macro F1, confusion matrices and learning curves.

We use built-in `TensorDataset` and `DataLoader` instead of a custom dataset class.

In [ ]:
from pathlib import Path
from collections import Counter
import copy
import os
import random
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from nltk.tokenize import TreebankWordTokenizer, TweetTokenizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

## 0.2 Locate the Lab 4 folder

**Google Colab:** place the notebook and both CSVs in `MyDrive/COMP4153/Lab4`.  
**Local Jupyter:** put them beside the notebook and open Jupyter from that folder.

The next cell selects the correct folder automatically. Change `COLAB_LAB_DIR` only if your Drive folder is different.

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    COLAB_LAB_DIR = Path("/content/drive/MyDrive/COMP4153/Lab4")
    LAB_DIR = COLAB_LAB_DIR
    print("Environment: Google Colab")
else:
    LAB_DIR = Path.cwd()
    print("Environment: local Jupyter Notebook")

if not LAB_DIR.exists():
    raise FileNotFoundError(f"Lab folder not found: {LAB_DIR}")

os.chdir(LAB_DIR)
print("Working directory:", Path.cwd())

## 0.3 Verify the two data files

Both data files should be present **before** running Part A or B. No dataset is downloaded in this notebook.

In [ ]:
required_files = [
    "imdb_reviews_10000_fixed_split.csv",
    "twitter_sentiment_fixed_split.csv",
]

for filename in required_files:
    print(f"{filename:40s}", Path(filename).exists())

missing = [f for f in required_files if not Path(f).exists()]

if missing:
    raise FileNotFoundError(
        "Missing required file(s): " + ", ".join(missing)
    )

## 0.4 Reproducibility and device

A random seed makes experiments easier to compare. PyTorch will use CUDA if available and otherwise use CPU. A GPU is **not required**; recurrent networks may simply train more slowly on a CPU.

In [ ]:
SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

---
# Part A — Guided IMDB sentiment classification

**Goal:** classify a complete movie review as positive or negative using word order.

```text
review → tokens → IDs [B,T] and lengths [B]
→ Embedding [B,T,D]
→ pack valid positions (ignore trailing <PAD>)
→ RNN / LSTM / GRU → final hidden state [B,H]
→ Dropout → Linear(H,2) → two sentiment logits [B,2]
```

For a simple RNN, a step can be written as $h_t = f(x_t, h_{t-1})$. LSTMs and GRUs introduce gates that regulate how information is retained or updated.

Part A is fully guided. Read the code comments and run each cell. The same helpers are reused for Twitter in Part B.

## A1. Load IMDB and use the supplied splits

The CSV includes `text`, `label` (`negative` / `positive`) and `split` (`train` / `dev` / `test`).

- **Training:** update model weights using mini-batches.
- **Development:** compare the completed epochs and select the checkpoint with the smallest development loss.
- **Test:** evaluate **after** checkpoint selection, not while choosing a model.

Keep the fixed split unchanged.

In [ ]:
imdb_df = pd.read_csv("imdb_reviews_10000_fixed_split.csv")

# Show some examples and verify the provided splits.
display(imdb_df.head(3))
display(pd.crosstab(imdb_df["split"], imdb_df["label"]))

imdb_train_df = imdb_df[imdb_df["split"] == "train"].reset_index(drop=True)
imdb_dev_df = imdb_df[imdb_df["split"] == "dev"].reset_index(drop=True)
imdb_test_df = imdb_df[imdb_df["split"] == "test"].reset_index(drop=True)

print("Train / development / test:", len(imdb_train_df), len(imdb_dev_df), len(imdb_test_df))

## A2. Tokenise and construct a training vocabulary

The model needs token **IDs**, not raw strings. Use the same lightweight IMDB tokeniser and vocabulary method as Lab 3:

```text
"This film was great!" → ["this", "film", "was", "great"]
→ vocabulary IDs
```

Reserve `PAD_IDX = 0` for right padding and `UNK_IDX = 1` for out-of-vocabulary words. Only IMDB **training reviews** contribute to `imdb_vocab`.

The vocabulary is shared across the three recurrent **cell types** for IMDB; we do *not* build a different vocabulary for RNN, LSTM or GRU.

In [ ]:
treebank_tokenizer = TreebankWordTokenizer()

def tokenize_imdb_nn(text):
    tokens = treebank_tokenizer.tokenize(
        str(text).lower()
    )

    return [
        token
        for token in tokens
        if len(token) > 1
        and any(ch.isalpha() for ch in token)
    ]

print(
    tokenize_imdb_nn(
        "This movie was not very good, but the ending was excellent!"
    )
)
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_IDX = 0
UNK_IDX = 1

def build_vocab(texts, tokenizer_fn, max_vocab=10000, min_freq=2):
    counter = Counter()

    for text in texts:
        counter.update(
            tokenizer_fn(text)
        )

    vocab = {
        PAD_TOKEN: PAD_IDX,
        UNK_TOKEN: UNK_IDX,
    }

    for token, count in counter.most_common(max_vocab):
        if (
            count >= min_freq
            and token not in vocab
        ):
            vocab[token] = len(vocab)

    return vocab, counter

imdb_vocab, imdb_counter = build_vocab(
    imdb_train_df["text"],
    tokenize_imdb_nn,
    max_vocab=10000,
    min_freq=2,
)

print("IMDB vocabulary size:", len(imdb_vocab))
print("20 most frequent training tokens:")
print(imdb_counter.most_common(20))

## A3. Encode reviews, including their true lengths

Sequence models need one extra item that the Lab 3 mean-pooling model did not use: the **number of valid tokens** in each padded review.

For a toy maximum length of 5:

```text
text:        "great movie"
input_ids:   [ID(great), ID(movie), 0, 0, 0]
length:      2
label:       1  (positive, if the review is positive)
```

Each IMDB review is **truncated or right-padded to `T = 300`**. The length is counted **after truncation**. An empty review is represented by one `<UNK>` token so that its length is at least 1 (a requirement of packed sequences).

`make_sequence_dataset()` returns a built-in `TensorDataset` storing **three** tensors:

| Tensor | Shape | Meaning |
|---|---|---|
| `input_ids` | `[N, 300]` | Retained word IDs followed by `<PAD>` |
| `lengths` | `[N]` | Valid token count (between 1 and 300) |
| `labels` | `[N]` | Negative `0`, positive `1` |

The same reusable helper works for short Twitter messages with `T = 40` in Part B.

In [ ]:
# Both datasets use the same binary sentiment label mapping.
LABEL_TO_ID = {"negative": 0, "positive": 1}
ID_TO_LABEL = {0: "negative", 1: "positive"}
IMDB_MAX_LEN = 300

# Convert text to IDs and return the number of real retained tokens.
def encode_sequence(text, tokenizer_fn, vocab, max_len):
    ids = [vocab.get(token, UNK_IDX) for token in tokenizer_fn(text)]
    ids = ids[:max_len]  # Truncate before counting the true length.

    if not ids:
        ids = [UNK_IDX]  # pack_padded_sequence() cannot process length 0.

    true_length = len(ids)
    padded_ids = ids + [PAD_IDX] * (max_len - true_length)
    return padded_ids, true_length

# Encode all texts once, then store (IDs, lengths, labels) as PyTorch tensors.
def make_sequence_dataset(dataframe, tokenizer_fn, vocab, max_len, text_column="text"):
    examples = [
        encode_sequence(text, tokenizer_fn, vocab, max_len)
        for text in dataframe[text_column]
    ]
    input_ids = torch.tensor([example[0] for example in examples], dtype=torch.long)
    lengths = torch.tensor([example[1] for example in examples], dtype=torch.long)
    labels = torch.tensor([LABEL_TO_ID[label] for label in dataframe["label"]], dtype=torch.long)
    return TensorDataset(input_ids, lengths, labels)

imdb_train_dataset = make_sequence_dataset(
    imdb_train_df,
    tokenize_imdb_nn,
    imdb_vocab,
    IMDB_MAX_LEN,
)
imdb_dev_dataset = make_sequence_dataset(
    imdb_dev_df,
    tokenize_imdb_nn,
    imdb_vocab,
    IMDB_MAX_LEN,
)
imdb_test_dataset = make_sequence_dataset(
    imdb_test_df,
    tokenize_imdb_nn,
    imdb_vocab,
    IMDB_MAX_LEN,
)

print("IMDB input IDs:", imdb_train_dataset.tensors[0].shape)
print("IMDB lengths:", imdb_train_dataset.tensors[1].shape)
print("IMDB labels:", imdb_train_dataset.tensors[2].shape)
print("First three lengths:", imdb_train_dataset.tensors[1][:3].tolist())

## A4. Make mini-batches

`make_loaders()` works for both datasets without changing its code. It batches the **three tensors together**, preserving the correspondence between a review, its true length and its label.

For a full IMDB training mini-batch (`B = 64`, `T = 300`):

```text
input_ids  [64, 300]
lengths    [64]
labels     [64]
```

Training mini-batches are shuffled. Development and test mini-batches are not shuffled, so their predictions remain aligned with the original DataFrame rows.

In [ ]:
def make_loaders(train_dataset, dev_dataset, test_dataset, train_batch_size, eval_batch_size=128, seed=SEED):
    generator = torch.Generator().manual_seed(
        seed
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        generator=generator,
    )

    dev_loader = DataLoader(
        dev_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=eval_batch_size,
        shuffle=False,
    )

    return (
        train_loader,
        dev_loader,
        test_loader,
    )

## A5. One classifier for **RNN, LSTM and GRU**

Rather than writing three classes, `RecurrentClassifier` accepts **one argument**, `cell_type`, which selects the PyTorch recurrent layer:

```python
CELL_TYPE = "rnn"   # Change to "lstm" or "gru" to try another model
```

All three models follow the same **many-to-one** pipeline. Here `V` is vocabulary size, `T` sequence length, `D` embedding width, `H` hidden width and `B` batch size.

| Stage | Layer / operation | Tensor shape |
|---|---|---|
| Input | Token IDs | `[B, T]` |
| Embedding table | `nn.Embedding(V, D)` | parameters `[V, D]` |
| Embedded words | Lookup each ID | `[B, T, D]` |
| Packing | `pack_padded_sequence(..., lengths)` | packed valid positions |
| Recurrent layer | `nn.RNN` / `nn.LSTM` / `nn.GRU` | final state `h_n`: `[1, B, H]` |
| Sequence representation | Select `h_n[-1]` | `[B, H]` |
| Dropout | `nn.Dropout(0.3)` | `[B, H]` |
| Output classifier | `nn.Linear(H, 2)` | **`[B, 2]` logits** |

**Why pack?** A shorter padded review must not let its final hidden state keep changing during `<PAD>` positions. `pack_padded_sequence()` uses the true lengths so the recurrent model stops at each example's last real token.

**RNN / GRU** return `(outputs, h_n)`. **LSTM** returns `(outputs, (h_n, c_n))`, where `c_n` is the additional cell-memory state. The classifier extracts `h_n` for all three, so the rest of the code is identical.

**Why no softmax in `forward()`?** `nn.CrossEntropyLoss` accepts raw logits and calculates the required log-probabilities internally.

In [ ]:
class RecurrentClassifier(nn.Module):
    def __init__(self, vocab_size, pad_idx=PAD_IDX, cell_type="gru", embedding_dim=32, hidden_dim=64, dropout=0.3):
        super().__init__()

        self.cell_type = cell_type.lower()

        # [V, D] trainable embedding table; row PAD_IDX does not learn.
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=pad_idx,
        )

        # One shared constructor works for all three sequence cells.
        cell_classes = {"rnn": nn.RNN, "lstm": nn.LSTM, "gru": nn.GRU}
        if self.cell_type not in cell_classes:
            raise ValueError("cell_type must be 'rnn', 'lstm', or 'gru'")

        self.recurrent = cell_classes[self.cell_type](
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_dim, 2)

    def forward(self, input_ids, lengths):
        # Token IDs [B,T] -> embedded word vectors [B,T,D].
        embedded = self.embedding(input_ids)

        # Strip right-padding for recurrent processing, using TRUE lengths.
        # Sorting is not needed because enforce_sorted=False.
        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )

        # LSTM has both hidden and cell memory; RNN/GRU have hidden state.
        _, state = self.recurrent(packed)
        h_n = state[0] if self.cell_type == "lstm" else state

        # Single-layer, forward-only model: h_n [1,B,H] -> [B,H].
        review_vector = h_n[-1]

        # Dropout then linear classifier: [B,H] -> [B,2] raw logits.
        return self.output(self.dropout(review_vector))

## A6. Training, evaluation and BPTT

For one labelled mini-batch, the essential PyTorch steps are:

```python
optimiser.zero_grad()                      # clear old gradients
logits = model(input_ids, lengths)         # forward sequence pass → [B, 2]
loss = loss_fn(logits, labels)             # cross-entropy vs correct labels
loss.backward()                            # backpropagation through time (BPTT)
torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)  # clip large gradients
optimiser.step()                           # update model parameters
```

`loss.backward()` propagates gradients through the output layer, recurrent operations and embeddings. RNNs can suffer from large (exploding) gradients, so **gradient clipping** limits their norm before the optimiser step.

`evaluate_model()` switches off dropout using `model.eval()` and disables gradients with `torch.no_grad()`. It returns loss, accuracy and all predictions/targets for later F1 and confusion-matrix evaluation.

The training helper completes **every configured epoch**, evaluates development data after each one, stores the **lowest-development-loss completed checkpoint**, and restores that checkpoint at the end. The test set is not consulted until final evaluation.

In [ ]:
def evaluate_model(model, data_loader, loss_fn, device):
    model.eval()  # Disable dropout during evaluation.
    total_loss = 0.0
    total_examples = 0
    correct = 0
    all_predictions = []
    all_targets = []

    with torch.no_grad():  # Evaluation never updates weights.
        for input_ids, lengths, labels in data_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            # lengths stays on CPU for pack_padded_sequence().
            logits = model(input_ids, lengths)  # [B,2] raw class scores.
            loss = loss_fn(logits, labels)
            predictions = logits.argmax(dim=1)
            batch_size = labels.size(0)

            # CrossEntropyLoss is a mean over this batch; weight it by size.
            total_loss += loss.item() * batch_size
            total_examples += batch_size
            correct += (predictions == labels).sum().item()
            all_predictions.extend(predictions.cpu().tolist())
            all_targets.extend(labels.cpu().tolist())

    return (
        total_loss / total_examples,
        correct / total_examples,
        all_predictions,
        all_targets,
    )

In [ ]:
def train_model(model, train_loader, dev_loader, loss_fn, optimiser, device, epochs=10, max_grad_norm=5.0):
    history = []  # One record per completed epoch for plotting.
    best_dev_loss = float("inf")
    best_dev_accuracy = -1.0
    best_epoch = None
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()  # Enable dropout during training.
        total_train_loss = 0.0
        total_train_examples = 0

        for input_ids, lengths, labels in train_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimiser.zero_grad()  # 1. Clear gradients left from previous batch.
            logits = model(input_ids, lengths)  # 2. Recurrent forward pass.
            loss = loss_fn(logits, labels)  # 3. Compare with sentiment labels.
            loss.backward()  # 4. BPTT: differentiate through token positions.

            # 5. Prevent excessively large gradient updates.
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=max_grad_norm,
            )
            optimiser.step()  # 6. Update embeddings, recurrence and classifier.

            batch_size = labels.size(0)
            total_train_loss += loss.item() * batch_size
            total_train_examples += batch_size

        train_loss = total_train_loss / total_train_examples
        dev_loss, dev_accuracy, _, _ = evaluate_model(
            model,
            dev_loader,
            loss_fn,
            device,
        )
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_loss": dev_loss,
            "dev_accuracy": dev_accuracy,
        })
        print(
            f"Epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} | "
            f"dev loss {dev_loss:.4f} | "
            f"dev accuracy {dev_accuracy:.3f}"
        )

        # Remember only the best COMPLETED epoch, but continue training.
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            best_dev_accuracy = dev_accuracy
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)  # Restore after ALL epochs finish.
    print(
        f"\nRestored epoch {best_epoch} | "
        f"best dev loss {best_dev_loss:.4f} | "
        f"dev accuracy {best_dev_accuracy:.3f}"
    )
    return pd.DataFrame(history), best_dev_loss, best_dev_accuracy, best_epoch

## A7. Train the IMDB sequence classifier

The **only line that selects the recurrent architecture** is:

```python
CELL_TYPE = "rnn"  # "rnn", "lstm", or "gru"
```

This model choice is used in **both** Part A and Part B. To compare recurrent architectures later, change `CELL_TYPE` here, then rerun the training/evaluation cells in both parts. The class and training helper stay unchanged.

Initial IMDB settings follow the uploaded Lab 4 recurrent classifier:

```text
D = 32 embeddings      H = 64 hidden units       dropout = 0.3
batch = 64             Adam learning rate = 0.001
weight decay = 1e-5    epochs = 10               gradient clipping = 5.0
```

Changing `CELL_TYPE` recreates **a fresh** model (after `set_seed()`) and a fresh optimiser. Training completes all epochs and restores the checkpoint selected by development loss. Running the full training on a CPU may take longer than the allotted lab period; focus on understanding the workflow first.

In [ ]:
# Change only CELL_TYPE to switch between the three recurrent architectures.
CELL_TYPE = "rnn"  # Supported: "rnn", "lstm", "gru".

# IMDB hyperparameters (from the recurrent classifier in Lab 4).
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 10
MAX_GRAD_NORM = 5.0

imdb_train_loader, imdb_dev_loader, imdb_test_loader = make_loaders(
    imdb_train_dataset,
    imdb_dev_dataset,
    imdb_test_dataset,
    train_batch_size=BATCH_SIZE,
)

# Reinitialise parameters so each model comparison begins afresh.
set_seed()
imdb_model = RecurrentClassifier(
    vocab_size=len(imdb_vocab),
    pad_idx=PAD_IDX,
    cell_type=CELL_TYPE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(device)

imdb_loss_fn = nn.CrossEntropyLoss()  # Logits [B,2] + class IDs [B].
imdb_optimiser = torch.optim.Adam(
    imdb_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

imdb_history, imdb_best_dev_loss, imdb_best_dev_accuracy, imdb_best_epoch = train_model(
    imdb_model,
    imdb_train_loader,
    imdb_dev_loader,
    imdb_loss_fn,
    imdb_optimiser,
    device,
    epochs=EPOCHS,
    max_grad_norm=MAX_GRAD_NORM,
)
print("Selected IMDB model:", CELL_TYPE.upper(), "| epoch:", imdb_best_epoch)

### Examine the learning curves

Plot **training vs development cross-entropy** and **development accuracy**. The first shows whether the model continues improving on unseen development examples; the second is an interpretable classification metric.

If training loss falls but development loss rises, the model may be overfitting. The training helper selects the lowest-development-loss completed epoch automatically.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(imdb_history["epoch"], imdb_history["train_loss"], marker="o", label="training loss")
ax.plot(imdb_history["epoch"], imdb_history["dev_loss"], marker="o", label="development loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.set_title(f"IMDB {CELL_TYPE.upper()}: training vs development loss")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(imdb_history["epoch"], imdb_history["dev_accuracy"], marker="o")
ax.set_xlabel("Epoch")
ax.set_ylabel("Development accuracy")
ax.set_ylim(0, 1)
ax.set_title(f"IMDB {CELL_TYPE.upper()}: development accuracy")
plt.show()

## A8. Evaluate the selected IMDB model on test data

Once **all** training epochs finish and the best development checkpoint has been restored, report:

- **Test loss:** average cross-entropy on unseen reviews;
- **Accuracy:** fraction classified correctly;
- **Macro F1:** average of the negative and positive class F1 scores;
- **Confusion matrix:** counts of predictions by actual/predicted class.

Use `argmax` on the two output logits to obtain the predicted class. The test split is not used to tune a model.

In [ ]:
imdb_test_loss, imdb_test_accuracy, imdb_test_predictions, imdb_test_targets = evaluate_model(
    imdb_model,
    imdb_test_loader,
    imdb_loss_fn,
    device,
)

imdb_test_f1 = f1_score(
    imdb_test_targets,
    imdb_test_predictions,
    average="macro",
)
print("IMDB model:", CELL_TYPE.upper())
print(f"Test loss:     {imdb_test_loss:.4f}")
print(f"Test accuracy: {imdb_test_accuracy:.3f}")
print(f"Macro F1:      {imdb_test_f1:.3f}")

imdb_cm = confusion_matrix(
    imdb_test_targets,
    imdb_test_predictions,
    labels=[0, 1],
)
imdb_disp = ConfusionMatrixDisplay(
    confusion_matrix=imdb_cm,
    display_labels=["negative", "positive"],
)
imdb_disp.plot()
plt.title(f"IMDB {CELL_TYPE.upper()}: confusion matrix")
plt.show()

## A9. Examine some classification mistakes

Accuracy summarises *how many* reviews are correct. The example mistakes may reveal *why* classification is difficult, including negation, mixed opinions, irony or reviews truncated to the first 300 tokens.

Predictions are aligned with the original test rows because the test DataLoader has `shuffle=False`.

In [ ]:
imdb_results = imdb_test_df.reset_index(drop=True).copy()
imdb_results["predicted_label"] = [
    ID_TO_LABEL[prediction]
    for prediction in imdb_test_predictions
]
imdb_errors = imdb_results[
    imdb_results["label"] != imdb_results["predicted_label"]
].copy()
imdb_errors["text_preview"] = (
    imdb_errors["text"]
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 300)
)
display(
    imdb_errors[["text_preview", "label", "predicted_label"]].head(5)
)

---
# Part B — Apply the **same RNN/LSTM/GRU classifier** to Twitter sentiment

Like Part A, this is **binary sentiment classification**, with one class label per tweet.

All reusable components are already defined:

```text
build_vocab()           → training-only vocabulary
encode_sequence()       → padded IDs and true sequence length
make_sequence_dataset() → TensorDataset(IDs, lengths, labels)
make_loaders()          → train/dev/test batches
RecurrentClassifier     → ONE implementation of RNN, LSTM and GRU
train_model()           → all epochs + best completed dev-loss checkpoint
evaluate_model()        → classification loss, accuracy and predictions
```

The Twitter dataset uses `clean_text` for model inputs and keeps the original `text` for error analysis. Tweets are shorter, so we use `T = 40` instead of `T = 300`.

Complete **three guided preparation steps** and **four tasks**. The architecture type is read from the existing `CELL_TYPE` variable in Part A: **do not rewrite the network or training functions**.

## Guided B1 — Load and inspect Twitter

Read the supplied `twitter_sentiment_fixed_split.csv` and inspect its fixed split counts. `text` is the original tweet and `clean_text` is the preprocessed text used as input.

Run the next cell without modifying it. The text-length histogram shows why a shorter maximum sequence length is appropriate.

In [ ]:
# Load and inspect the supplied Twitter split.
twitter_df = pd.read_csv(
    "twitter_sentiment_fixed_split.csv"
)

print("Shape:", twitter_df.shape)
display(twitter_df.head(3))

print("\nSplit × label counts:")
display(
    pd.crosstab(
        twitter_df["split"],
        twitter_df["label"],
    )
)

print("\nMissing values:")
print(
    twitter_df[
        [
            "text",
            "clean_text",
            "label",
            "split",
        ]
    ].isna().sum()
)

twitter_df["clean_length"] = (
    twitter_df["clean_text"]
    .astype(str)
    .str.split()
    .str.len()
)

twitter_df["clean_length"].plot(
    kind="hist",
    bins=30,
    figsize=(7, 4),
    title="Twitter clean-text length distribution",
)

plt.xlabel(
    "Cleaned whitespace-separated tokens"
)
plt.show()

## Guided B2 — Tokenise Twitter and build its vocabulary

Reuse `build_vocab()` exactly as in Part A, but with `TweetTokenizer` and the **training tweets' `clean_text`** column. The word vocabulary will differ from IMDB because the writing style and dataset are different.

We do **not** use Twitter development or test text to construct its vocabulary.

In [ ]:
# Use the supplied split exactly as stored.
twitter_train_df = (
    twitter_df[
        twitter_df["split"] == "train"
    ]
    .reset_index(drop=True)
)

twitter_dev_df = (
    twitter_df[
        twitter_df["split"] == "dev"
    ]
    .reset_index(drop=True)
)

twitter_test_df = (
    twitter_df[
        twitter_df["split"] == "test"
    ]
    .reset_index(drop=True)
)

print("Train:", twitter_train_df.shape)
print("Dev:  ", twitter_dev_df.shape)
print("Test: ", twitter_test_df.shape)

tweet_tokenizer = TweetTokenizer(
    preserve_case=False,
    reduce_len=True,
)

word_pattern = re.compile(
    r"[a-z]+(?:'[a-z]+)?"
)

def tokenize_twitter_nn(text):
    tokens = tweet_tokenizer.tokenize(
        str(text)
    )

    return [
        token
        for token in tokens
        if word_pattern.fullmatch(token)
    ]

twitter_vocab, twitter_counter = build_vocab(
    twitter_train_df["clean_text"],
    tokenize_twitter_nn,
    max_vocab=10000,
    min_freq=2,
)

print(
    "Twitter vocabulary size:",
    len(twitter_vocab),
)
print(
    "20 most frequent training tokens:"
)
print(
    twitter_counter.most_common(20)
)

## Guided B3 — Create Twitter tensors containing IDs, lengths and labels

Reuse `make_sequence_dataset()` from Part A, this time with:

```text
text column = clean_text
T = TWITTER_MAX_LEN = 40
```

The three tensors per example are:

```text
input_ids [N, 40]
lengths   [N]
labels    [N]
```

The existing `make_loaders()` function will batch them automatically.

In [ ]:
TWITTER_MAX_LEN = 40

# Identical encoding/dataset helper: only the text column and maximum length differ.
twitter_train_dataset = make_sequence_dataset(
    twitter_train_df,
    tokenize_twitter_nn,
    twitter_vocab,
    TWITTER_MAX_LEN,
    text_column="clean_text",
)
twitter_dev_dataset = make_sequence_dataset(
    twitter_dev_df,
    tokenize_twitter_nn,
    twitter_vocab,
    TWITTER_MAX_LEN,
    text_column="clean_text",
)
twitter_test_dataset = make_sequence_dataset(
    twitter_test_df,
    tokenize_twitter_nn,
    twitter_vocab,
    TWITTER_MAX_LEN,
    text_column="clean_text",
)
print("Twitter input IDs:", twitter_train_dataset.tensors[0].shape)
print("Twitter true lengths:", twitter_train_dataset.tensors[1].shape)
print("Twitter labels:", twitter_train_dataset.tensors[2].shape)

## Task B1 — Train a Twitter **RNN/LSTM/GRU** sentiment classifier

The data and helper functions are already provided. Complete the same four steps demonstrated in **A7**:

1. Create a **new** `RecurrentClassifier` with `vocab_size=len(twitter_vocab)` and **`cell_type=CELL_TYPE`**.
2. Define `twitter_loss_fn = nn.CrossEntropyLoss()`.
3. Create an Adam optimiser using the provided learning rate and weight decay.
4. Call **`train_model()`** to finish all epochs, record `twitter_history`, and restore the lowest-development-loss completed checkpoint.

The starting Twitter settings are `D=32`, `H=32`, dropout `0.3`, batch size `64`, learning rate `0.001`, weight decay `1e-5`, `10` epochs and gradient clip `5.0`.

**Switch models using just the `CELL_TYPE` setting in A7** (`"rnn"`, `"lstm"`, `"gru"`). Rerun this Twitter training cell after changing it. There are no separate implementations for the three models.

In [ ]:
# Task B1 — reuse the exact same sequence-model class and trainer from Part A.
# Change only CELL_TYPE to switch between the three recurrent architectures.
CELL_TYPE = "rnn"  # Supported: "rnn", "lstm", "gru".
EMBEDDING_DIM = 32
HIDDEN_DIM = 32
DROPOUT = 0.3
BATCH_SIZE = 64
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 10
MAX_GRAD_NORM = 5.0

twitter_train_loader, twitter_dev_loader, twitter_test_loader = make_loaders(
    twitter_train_dataset,
    twitter_dev_dataset,
    twitter_test_dataset,
    train_batch_size=BATCH_SIZE,
)
set_seed()

# Follow A7 and complete these four steps:
# 1. Create twitter_model = RecurrentClassifier(..., cell_type=CELL_TYPE, ...).to(device).
# 2. Create twitter_loss_fn with CrossEntropyLoss.
# 3. Create a fresh Adam optimiser for twitter_model.
# 4. Call train_model(..., epochs=EPOCHS, max_grad_norm=MAX_GRAD_NORM)
#    and save twitter_history, twitter_best_dev_loss,
#    twitter_best_dev_accuracy and twitter_best_epoch.

# YOUR CODE HERE

## Task B2 — Plot learning curves for the Twitter sequence classifier

Use `twitter_history` from B1 to make two plots:

1. **Training and development cross-entropy** against epoch on the **same** figure.
2. **Development accuracy** against epoch on a second figure.

Refer to the IMDB plots in **A7**. Add axis labels, titles and a legend. Compare the development-loss minimum with `twitter_best_epoch`. The test split is not used for these plots.

In [ ]:
# Task B2 — plot training vs development loss and development accuracy.
# Follow the two IMDB plotting examples in A7.
# Data columns: epoch, train_loss, dev_loss, dev_accuracy.
# Make separate figures, add titles, axis labels and a legend where needed.

# YOUR CODE HERE

## Task B3 — Evaluate the selected Twitter model on the **test** set

The evaluation helper is already called for you. Complete the remaining calculations:

- calculate and print **macro F1** alongside loss and accuracy;
- show a **confusion matrix** with classes `negative` and `positive`.

Follow the IMDB example in **A8**. The selected checkpoint comes only from development loss; do not choose or tune the model using the Twitter test results.

In [ ]:
# The test DataLoader does not shuffle, so predictions match test rows.
twitter_test_loss, twitter_test_accuracy, twitter_test_predictions, twitter_test_targets = evaluate_model(
    twitter_model,
    twitter_test_loader,
    twitter_loss_fn,
    device,
)

# Complete the evaluation using A8 as an example:
# 1. Calculate macro F1 with f1_score(..., average="macro").
# 2. Print test loss, accuracy and macro F1.
# 3. Plot ConfusionMatrixDisplay for negative and positive classes.

# YOUR CODE HERE

## Task B4 — Inspect incorrect Twitter sentiment predictions

Follow the IMDB example in **A9**:

1. Add the predicted label strings (`negative` / `positive`) to the test DataFrame using `ID_TO_LABEL`.
2. Select rows where `label != predicted_label`.
3. Display the first **eight** mistakes with `text`, `clean_text`, `label`, and `predicted_label`.

**Think about:** negation, emoji removed during cleaning, sarcasm, and missing conversational context. The purpose is to diagnose possible weaknesses, not to reselect a model using the test set.

In [ ]:
# Predictions align with rows because twitter_test_loader uses shuffle=False.
twitter_results = twitter_test_df.reset_index(drop=True).copy()

# Use the Part A error-analysis cell A9 as your guide.
# 1. Add predicted_label by mapping predictions through ID_TO_LABEL.
# 2. Filter rows where true and predicted sentiment differ.
# 3. Display eight incorrect predictions and the original/cleaned text.

# YOUR CODE HERE

## Guided demonstration — Sequence order now changes the hidden state

Mean pooling (used in Lab 3) gives the **same vector** for the same words in any order. A recurrent network processes the words in sequence, so its **final hidden state can change** when their order is reversed.

The code below uses three **distinct, known training-vocabulary words**. It compares the average embeddings with the output of the *trained* Twitter recurrent layer. It does **not** prove that the model understands these artificial word sequences or that their sentiment should differ.

The model weights are unchanged; the demonstration uses `eval()` and `torch.no_grad()`.

In [ ]:
# Choose three distinct known words so the comparison is not just <UNK> tokens.
example_words = [
    word for word in ["good", "day", "bad", "love", "happy", "sad"]
    if word in twitter_vocab
]
if len(example_words) < 3:
    example_words = [
        word for word in twitter_vocab
        if word not in {PAD_TOKEN, UNK_TOKEN}
    ][:3]
assert len(example_words) >= 3, "Need three distinct known tokens"
example_words = example_words[:3]

forward_words = example_words
reverse_words = list(reversed(example_words))
print("Sequence A:", forward_words)
print("Sequence B:", reverse_words)

order_ids = torch.tensor(
    [
        [twitter_vocab[word] for word in forward_words],
        [twitter_vocab[word] for word in reverse_words],
    ],
    dtype=torch.long,
).to(device)
order_lengths = torch.tensor([3, 3], dtype=torch.long)

twitter_model.eval()
with torch.no_grad():
    embedded = twitter_model.embedding(order_ids)  # [2, 3, D]
    mean_vectors = embedded.mean(dim=1)  # Order-independent average [2, D].
    packed = pack_padded_sequence(
        embedded,
        order_lengths,
        batch_first=True,
        enforce_sorted=False,
    )
    _, state = twitter_model.recurrent(packed)
    hidden = state[0] if twitter_model.cell_type == "lstm" else state
    final_vectors = hidden[-1]  # Order-sensitive final states [2, H].
    logits = twitter_model(order_ids, order_lengths)
    probs = torch.softmax(logits, dim=1)[:, 1]

print("Mean embedding L2 distance:", torch.norm(mean_vectors[0] - mean_vectors[1]).item())
print("Final hidden-state L2 distance:", torch.norm(final_vectors[0] - final_vectors[1]).item())
print("P(positive), sequence A:", round(probs[0].item(), 4))
print("P(positive), sequence B:", round(probs[1].item(), 4))

# Wrap-up

Both parts used the **same binary text-classification workflow**:

```text
raw text → tokens → training-only vocabulary → padded IDs + true lengths
→ trainable embeddings → packed RNN / LSTM / GRU
→ final hidden state → dropout → 2 class logits
→ cross-entropy → BPTT + gradient clipping → Adam
→ lowest-development-loss completed checkpoint
→ test accuracy, macro F1, confusion matrix and error analysis
```

**What changes between Part A and Part B?** The dataset, tokenizer, vocabulary, input sequence length and starting hyperparameters. **What stays the same?** `RecurrentClassifier`, `make_sequence_dataset()`, `make_loaders()`, `train_model()` and `evaluate_model()`.

**Model comparison:** Change only `CELL_TYPE` in A7 to `"rnn"`, `"lstm"` or `"gru"`; rerun the training/evaluation cells with a new model and optimiser. Compare **development** results for model choice, not repeated test-set results.

**Key ideas:**

- Sequence order can affect recurrent hidden states; mean embedding pooling is order-independent.
- `pack_padded_sequence()` makes the final state correspond to each text's **last valid token**.
- RNN, LSTM and GRU use the same outer interface; LSTM additionally returns a cell-memory state.
- Training updates weights with BPTT; gradient clipping controls large gradients.
- Train for **all** epochs, then restore the best completed development-loss checkpoint.
- Classify using two raw logits and evaluate with loss, accuracy, macro F1 and mistakes.

Save the notebook after finishing Tasks B1–B4.